<a href="https://colab.research.google.com/github/Tahir-yamin/recursive-autonomy-research/blob/main/colab/RAR_Colab_Campaign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAR v2 — Pre-Registered Campaign on Google Colab (Ollama + gemma2:9b)

Self-contained run of the pre-registered extended evaluation (`PREREGISTRATION.md`): **N=10 seeds × 60 cycles**, three conditions, with Phase-2 instrumentation. Mirrors the Kaggle run for cross-verification.

## Before you run
1. **Runtime → Change runtime type → GPU (T4)**
2. In **cell 6**, set `TASK = 'A'` (synthetic) or `'B'` (CIFAR-10→PCA64). One run per task.
3. **Runtime → Run all.** ≈ 5–7 h per task (keep the tab active; Colab disconnects on idle).

Output: `pilot_results_task{A|B}.json` is offered for download at the end — send it back.

## 1. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Install Ollama + start server

In [ ]:
import os, time, subprocess, urllib.request, shutil
os.system('apt-get -qq update >/dev/null 2>&1 && apt-get -qq install -y zstd >/dev/null 2>&1')
rc = os.system('curl -fsSL https://ollama.com/install.sh | sh')
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert shutil.which('ollama') or os.path.exists(ollama_bin), f'Ollama install failed (rc={rc}).'
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
server = subprocess.Popen([ollama_bin,'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3); print('Ollama is up.'); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start.')

## 3. Pull the orchestrator model (gemma2:9b)

In [ ]:
import os
assert os.system(f'{ollama_bin} pull gemma2:9b') == 0, 'gemma2:9b pull failed'
MODEL = 'gemma2:9b'; print('USING MODEL:', MODEL)

## 4. Clone repo + install deps

In [ ]:
import os
os.system('pip -q install aiohttp networkx scikit-learn scipy matplotlib nest_asyncio >/dev/null 2>&1')
os.system('rm -rf /content/rar && git clone --depth 1 https://github.com/Tahir-yamin/recursive-autonomy-research /content/rar')
print(os.listdir('/content/rar'))

## 5. Configure backend + health check (one real JSON call)

In [ ]:
import os, sys, importlib, asyncio, nest_asyncio
nest_asyncio.apply()
os.environ['LLM_BASE_URL'] = 'http://127.0.0.1:11434/v1/chat/completions'
os.environ['LLM_API_KEY'] = 'ollama'
os.environ['OPENROUTER_MODEL'] = MODEL
os.environ['OPENROUTER_MAX_TOKENS'] = '4000'
os.environ['RAR_TORCH_DEVICE'] = 'cpu'  # match Kaggle for cross-verification; LLM is the bottleneck
os.chdir('/content/rar'); sys.path.insert(0, '/content/rar')
import run_pilot_experiment as rp; importlib.reload(rp)
async def health():
    r = await rp.call_llm(rp.SEARCH_SPACE_PROMPT + '\nPropose a config. Respond ONLY JSON. First trial.')
    cfg = rp.parse_json_response(r) if r else None
    print('PARSED:', cfg, '| VALID:', rp.is_valid_config(cfg) if cfg else False)
    return rp.is_valid_config(cfg) if cfg else False
assert asyncio.run(health()), 'Health check failed.'

## 6. Run the pre-registered campaign
Set `TASK` to `'A'` (synthetic manifold) or `'B'` (CIFAR-10→PCA64). One run per task.

In [ ]:
import os, asyncio, importlib, nest_asyncio, run_pilot_experiment as rp
nest_asyncio.apply()
TASK = 'A'   # <<< SET 'A' or 'B'
os.environ['TASK'] = TASK
os.environ['RAR_CYCLES'] = '60'
os.environ.pop('RAR_SEEDS', None)
importlib.reload(rp)
asyncio.run(rp.execute_campaign())
print(f'CAMPAIGN COMPLETE (TASK={TASK})')

## 7. Summary, figure, and download

In [ ]:
import json, shutil, os, numpy as np
res = json.load(open('/content/rar/pilot_results.json'))
print('TASK:', os.environ.get('TASK'), '| SEEDS:', res['SEEDS'])
print('Wilcoxon p (RAR vs Baseline):', res['wilcoxon_p_value_RAR_vs_Baseline'])
for c in ['stateless_baseline','vector_rag','rar_compressed']:
    cd = res['data']['conditions'][c]; ta = np.array(cd['test_accuracies'])*100
    bf = cd.get('best_found_trajectories', []); fin = np.array([t[-1] for t in bf])*100 if bf else None
    ctx = cd.get('best_in_context_trajectories', [])
    vis = np.mean([np.mean(t[-10:]) for t in ctx])*100 if ctx else float('nan')
    llm = sum(cd.get('llm_proposal_counts', [])); heu = sum(cd.get('heuristic_proposal_counts', []))
    ex = f'  best-found(final)={fin.mean():.2f}%' if fin is not None else ''
    print(f'{c:18s} test={ta.mean():.2f}±{ta.std():.2f}%{ex}  late best-visible={vis:.0f}%  LLM={llm} heur={heu}')
os.system('cd /content/rar && python plot_best_found.py')
T = os.environ.get('TASK','A')
for s,dst in [('pilot_results.json',f'pilot_results_task{T}.json'),('fig_best_found_vs_cycle.png',f'fig_best_found_vs_cycle_task{T}.png')]:
    if os.path.exists(f'/content/rar/{s}'): shutil.copy(f'/content/rar/{s}', f'/content/{dst}')
try:
    from google.colab import files
    files.download(f'/content/pilot_results_task{T}.json')
except Exception as e:
    print('Download manually from the Files panel:', e)